In [39]:
import torch
import torch.nn as nn
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

In [40]:
class MyBatchNormAnyd(nn.Module):
    def __init__(self, num_fetures, mometum = 0.1, epsilon = 1e-5):
        super().__init__()
        self.num_fetures = num_fetures
        self.mometum = mometum
        self.epsilon = epsilon
        self.weight = nn.Parameter(torch.ones(num_fetures))
        self.bias = nn.Parameter(torch.zeros(num_fetures))

        self.register_buffer('running_mean', torch.zeros(num_fetures))
        self.register_buffer('running_var', torch.ones(num_fetures))
        self.training = True

    def forward(self, X):
        if self.training:
            mean = X.transpose(0, 1).reshape(self.num_fetures, -1).mean(dim = 1)

            var = X.transpose(0, 1).reshape(self.num_fetures, -1).var(dim = 1)

            self.running_mean = self.mometum * mean + (1 - self.mometum) * self.running_mean
            self.running_var = self.mometum * var + (1 - self.mometum) * self.running_var

        else:
            mean = self.running_mean
            var = self.running_var

        X_normalized = (X - mean.reshape(-1, * [1] * (X.dim() - 1)).transpose(0,1)) / torch.sqrt(var.reshape(-1,* [1] * (X.dim() - 1)).transpose(0,1) + self.epsilon)

        return X_normalized * self.weight.reshape(-1, * [1] * (X.dim() - 1)).transpose(0,1) + self.bias.reshape(-1, * [1] * (X.dim() - 1)).transpose(0,1)


In [41]:
num_fetures = 5
spacial_size = [20, 10, 15]

train_size = 20
test_size = 20

X_train = torch.randn(train_size, num_fetures, *spacial_size).to(device)

X_test = torch.randn(test_size, num_fetures, *spacial_size).to(device)
alpha = 0.9
epsilon = 1e-5

In [42]:
my_model = MyBatchNormAnyd(num_fetures, 1 - alpha).to(device)

In [43]:
if len(spacial_size) <= 1:
    model = nn.BatchNorm1d(num_fetures, epsilon, 1-alpha)
if len(spacial_size) == 2:
    model = nn.BatchNorm2d(num_fetures, epsilon, 1-alpha)
if len(spacial_size) == 3:
    model = nn.BatchNorm3d(num_fetures, epsilon, 1-alpha)
model.to(device)

BatchNorm3d(5, eps=1e-05, momentum=0.09999999999999998, affine=True, track_running_stats=True)

In [44]:
num_iterations = 10

running_mean_gap = torch.empty(num_iterations)
running_var_gap = torch.empty(num_iterations)

model.train()
my_model.train()

for i in range(num_iterations):
    model(X_train)
    my_model.forward(X_train)
    running_mean_gap[i] = torch.max(torch.abs(model.running_mean - my_model.running_mean))
    running_var_gap[i] = torch.max(torch.abs(model.running_var - my_model.running_var))

    print(running_mean_gap)
    print(running_var_gap)

tensor([5.8208e-11, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.3298e-37, 2.3822e-44,
        4.1326e-39, 3.3061e-39, 7.8473e-43, 0.0000e+00])
tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0018, 0.0016, 0.0015, 0.0013, 0.0012,
        0.0011])
tensor([5.8208e-11, 1.1642e-10, 0.0000e+00, 0.0000e+00, 1.3298e-37, 2.3822e-44,
        4.1326e-39, 3.3061e-39, 7.8473e-43, 0.0000e+00])
tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0018, 0.0016, 0.0015, 0.0013, 0.0012,
        0.0011])
tensor([5.8208e-11, 1.1642e-10, 2.3283e-10, 0.0000e+00, 1.3298e-37, 2.3822e-44,
        4.1326e-39, 3.3061e-39, 7.8473e-43, 0.0000e+00])
tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0018, 0.0016, 0.0015, 0.0013, 0.0012,
        0.0011])
tensor([5.8208e-11, 1.1642e-10, 2.3283e-10, 2.3283e-10, 1.3298e-37, 2.3822e-44,
        4.1326e-39, 3.3061e-39, 7.8473e-43, 0.0000e+00])
tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0018, 0.0016, 0.0015, 0.0013, 0.0012,
        0.0011])
tensor([5.8208e-11, 1.1642e-10, 2.3283e-10, 2.3283e-10, 4.6566e-

In [45]:
model.eval()
my_model.eval()

y_test = model(X_test)
y_my_test = my_model(X_test)

print(torch.max(torch.abs(y_test - y_my_test)).item())

4.76837158203125e-07
